# Profiles to HEALPix (Batch)

Convert a **batch** of EarthCARE atmospheric profile segments (ATLID lidar, CPR radar) to HEALPix:

- 1D lat/lon along the ground track
- A vertical dimension (height) with atmospheric columns
- Each profile is mapped to a HEALPix cell, the vertical axis is preserved

Each segment (identified by its `orbit_and_frame`, e.g. `01164E`) is loaded, converted, and written to its own HEALPix DGGS Zarr store independently. If one segment fails (e.g. missing file, bad data), it is skipped and reported at the end instead of stopping the whole batch.

In [ ]:
import earthcarekit as eck
import healpix_geo as hpxg
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import xdggs
import zarr

import cartopy.crs as ccrs
import cartopy.feature as cfeature


from earthcare_dggs.convert import lonlat_to_healpix_cells
from earthcare_dggs.settings import ATLID_DEPTH, ELLIPSOID

## Input parameters

`ORBITS` now takes a **list** of `orbit_and_frame` values instead of a single one. Add as many segments as you like.

In [ ]:
PRODUCT = "ACM_CAP_2B"
ORBITS = [
    "01164E",
    "01164F",
    "01165A",
    # ... add as many segments as needed
]
OUTPUT_PATH = ""

## Conversion pipeline

The single-segment workflow from the original notebook is split into three steps:

1. `load_segment` – find and load the product file for one segment
2. `add_healpix_cell_ids` – map along-track profiles to HEALPix cells
3. `save_to_dggs_zarr` – encode with `xdggs` and write a DGGS-convention Zarr store

`convert_segment_to_healpix` chains these three steps for a single segment. The batch loop further down calls it once per segment in `ORBITS`.

In [ ]:
def load_segment(product, orbit_and_frame):
    """Find and load one EarthCARE segment as an xarray Dataset."""
    result = eck.search_product(file_type=product, orbit_and_frame=orbit_and_frame)
    with eck.read_product(result.filepath[0]) as ds:
        profile = ds.load()

    print(f"[{orbit_and_frame}] Dimensions: {dict(profile.sizes)}")
    print(f"[{orbit_and_frame}] Lat: [{float(profile['latitude'].min()):.1f}, {float(profile['latitude'].max()):.1f}]")
    print(f"[{orbit_and_frame}] Lon: [{float(profile['longitude'].min()):.1f}, {float(profile['longitude'].max()):.1f}]")
    return profile


def add_healpix_cell_ids(profile, orbit_and_frame):
    """Assign each along-track profile to a HEALPix cell; the vertical axis is preserved as-is."""
    prof_cell_ids = lonlat_to_healpix_cells(
        profile["longitude"].values, profile["latitude"].values,
        depth=ATLID_DEPTH, ellipsoid=ELLIPSOID,
    )

    unique_cells = np.unique(prof_cell_ids)
    print(f"[{orbit_and_frame}] HEALPix depth {ATLID_DEPTH}: {len(profile.along_track)} profiles → {len(unique_cells)} unique cells")
    print(f"[{orbit_and_frame}] Average profiles per cell: {len(profile.along_track) / len(unique_cells):.1f}")

    # Add cell_ids as coordinate
    profile_healpix = profile.assign_coords(cell_ids=("along_track", prof_cell_ids))
    profile_healpix.attrs.update({
        "source": "EarthCARE MSI L2A",
        "healpix_level": ATLID_DEPTH,
        "healpix_indexing": "nested",
        "healpix_ellipsoid": ELLIPSOID,
        "orbit_and_frame": orbit_and_frame,
    })
    return profile_healpix


def save_to_dggs_zarr(profile_healpix, product, orbit_and_frame, output_path):
    """Encode with xdggs and write a DGGS-convention Zarr store for one segment."""
    # Add xdggs index
    profile_healpix = xdggs.decode(
        profile_healpix,
        grid_info=xdggs.HealpixInfo(level=ATLID_DEPTH, indexing_scheme="nested"),
    )

    # Encode with xdggs convention
    profile_encoded = xdggs.encode(profile_healpix, "xdggs")

    # Write to Zarr (one store per segment)
    segment_output_path = f"{output_path}earthcare_{product}_{orbit_and_frame}.zarr"
    profile_encoded.to_zarr(segment_output_path, mode="w", consolidated=True)

    # Add DGGS Zarr convention metadata (compatible with legacy-converters format)
    dggs_convention = {
        "uuid": "7b255807-140c-42ca-97f6-7a1cfecdbc38",
        "name": "dggs",
        "schema_url": "https://raw.githubusercontent.com/zarr-conventions/dggs/refs/tags/v1/schema.json",
        "spec_url": "https://github.com/zarr-conventions/dggs/blob/v1/README.md",
        "description": "Discrete Global Grid Systems convention for zarr",
    }
    dggs_meta = {
        "name": "healpix",
        "refinement_level": ATLID_DEPTH,
        "indexing_scheme": "nested",
        "ellipsoid": {
            "name": "wgs84",
            "semimajor_axis": 6378137.0,
            "inverse_flattening": 298.257223563,
        },
        "spatial_dimension": "cell_ids",
        "coordinate": "cell_ids",
        "compression": "none",
    }

    root = zarr.open_group(segment_output_path, mode="r+")
    root.attrs["zarr_conventions"] = [dggs_convention]
    root.attrs["dggs"] = dggs_meta

    print(f"[{orbit_and_frame}] Saved to {segment_output_path}")
    return segment_output_path


def convert_segment_to_healpix(product, orbit_and_frame, output_path=""):
    """Run the full load -> HEALPix -> save pipeline for a single EarthCARE segment."""
    profile = load_segment(product, orbit_and_frame)
    profile_healpix = add_healpix_cell_ids(profile, orbit_and_frame)
    return save_to_dggs_zarr(profile_healpix, product, orbit_and_frame, output_path)

## Run batch conversion

Loops over `ORBITS`, converting each segment in turn. A failure on one segment (e.g. a missing file or a bad download) is caught and logged so the rest of the batch keeps running.

In [ ]:
results = {}
for orbit_and_frame in ORBITS:
    print(f"=== Processing {orbit_and_frame} ===")
    try:
        path = convert_segment_to_healpix(PRODUCT, orbit_and_frame, OUTPUT_PATH)
        results[orbit_and_frame] = {"status": "ok", "path": path}
    except Exception as e:
        print(f"[{orbit_and_frame}] FAILED: {e}")
        results[orbit_and_frame] = {"status": "failed", "error": str(e)}
    print()

## Summary

In [ ]:
n_ok = sum(1 for r in results.values() if r["status"] == "ok")
n_failed = len(results) - n_ok

print(f"Batch complete: {n_ok}/{len(results)} segment(s) converted successfully.")

if n_ok:
    print("\nSuccessful segments:")
    for orbit_and_frame, r in results.items():
        if r["status"] == "ok":
            print(f"  {orbit_and_frame}: {r['path']}")

if n_failed:
    print("\nFailed segments:")
    for orbit_and_frame, r in results.items():
        if r["status"] == "failed":
            print(f"  {orbit_and_frame}: {r['error']}")